# Async Python — Senior Python Interview Practice\n\nFour interview-style exercises spanning implementation, trade-offs, and production concerns.\n\n**How to use:** attempt each prompt first, then run and critique the reference solution. Discuss trade-offs aloud as you would in a senior-level interview.


## 1. Concurrent fetches\n\n### Problem statement\nFetch URLs concurrently while limiting concurrency.


In [ ]:
import asyncio

async def bounded_map(items, worker, limit=10):
    sem = asyncio.Semaphore(limit)
    async def run(item):
        async with sem: return await worker(item)
    return await asyncio.gather(*(run(item) for item in items))


### Complexity\n- **Time:** Roughly max individual latency at sufficient concurrency\n- **Space:** O(number of tasks)\n\n### Interview tip\nBound concurrency; `gather` alone can overwhelm a downstream service.\n\n### Follow-up questions\n- How do you handle partial failures and timeouts?


## 2. Async retry\n\n### Problem statement\nRetry an async function with exponential backoff and cancellation propagation.


In [ ]:
import asyncio

async def async_retry(operation, attempts=3, base_delay=.1):
    for attempt in range(attempts):
        try: return await operation()
        except asyncio.CancelledError: raise
        except Exception:
            if attempt == attempts - 1: raise
            await asyncio.sleep(base_delay * 2 ** attempt)


### Complexity\n- **Time:** O(attempts × operation cost)\n- **Space:** O(1)\n\n### Interview tip\nNever accidentally swallow `CancelledError`.\n\n### Follow-up questions\n- Add jitter and a retryability predicate for HTTP status codes.


## 3. Producer consumer\n\n### Problem statement\nProcess jobs with workers and sentinels using `asyncio.Queue`.


In [ ]:
import asyncio

async def worker(queue, handle):
    while True:
        item = await queue.get()
        try:
            if item is None: return
            await handle(item)
        finally:
            queue.task_done()


### Complexity\n- **Time:** O(jobs)\n- **Space:** O(queue maxsize)\n\n### Interview tip\nA bounded queue provides backpressure; always pair `get` with `task_done`.\n\n### Follow-up questions\n- How would you collect worker exceptions and shut down cleanly?


## 4. Async timeout\n\n### Problem statement\nCall an async operation with a deadline and return a fallback on timeout.


In [ ]:
import asyncio

async def with_timeout(operation, seconds, fallback=None):
    try:
        async with asyncio.timeout(seconds):
            return await operation()
    except TimeoutError:
        return fallback


### Complexity\n- **Time:** Operation cost up to deadline\n- **Space:** O(1)\n\n### Interview tip\nClarify whether timeout should cancel work or merely stop waiting; this cancels the task scope.\n\n### Follow-up questions\n- How do nested time budgets propagate through a request?
